# Hyperparameter Tuning for eBay Price Prediction

This notebook performs hyperparameter tuning on XGBoost for eBay price prediction.

Steps:
1. Load processed data and feature metadata
2. Set up XGBoost hyperparameter grid
3. Perform randomized search with cross-validation
4. Evaluate tuned model on test set
5. Compare tuned vs baseline performance
6. Save best model parameters

In [8]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Set random seed for reproducibility
np.random.seed(42)

In [9]:
# Load processed data and feature metadata
processed_dir = Path('../datasets/processed')
train_path = processed_dir / 'ebay_train_processed.csv'
test_path = processed_dir / 'ebay_test_processed.csv'
feature_info_path = processed_dir / 'feature_info.json'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
with open(feature_info_path, 'r') as f:
    feature_info = json.load(f)

print(f'Train samples: {len(train_df):,}')
print(f'Test samples: {len(test_df):,}')
print(f'Available numeric features: {len(feature_info["numeric_features"]):,}')

Train samples: 5,406
Test samples: 1,352
Available numeric features: 26


In [10]:
# Select features for modeling
scaled_features = feature_info['scaled_features']
additional_features = ['category_encoded', 'brand_encoded', 'group_count', 'condition_bucket']
feature_columns = scaled_features + additional_features
target_column = 'price_log'

print('Using features:')
print(feature_columns)

X_train = train_df[feature_columns]
y_train = train_df[target_column]
X_test = test_df[feature_columns]
y_test = test_df[target_column]

Using features:
['title_length_scaled', 'seller_feedback_score_scaled', 'seller_feedback_pct_scaled', 'group_median_price_scaled', 'group_mean_price_scaled', 'group_std_price_scaled', 'title_emb_0_scaled', 'title_emb_1_scaled', 'title_emb_2_scaled', 'title_emb_3_scaled', 'title_emb_4_scaled', 'title_emb_5_scaled', 'title_emb_6_scaled', 'title_emb_7_scaled', 'title_emb_8_scaled', 'title_emb_9_scaled', 'title_emb_10_scaled', 'title_emb_11_scaled', 'title_emb_12_scaled', 'title_emb_13_scaled', 'title_emb_14_scaled', 'title_emb_15_scaled', 'category_encoded', 'brand_encoded', 'group_count', 'condition_bucket']


In [11]:
# Define XGBoost hyperparameter grid
xgb_param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 6, 9, 12],
    'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.3],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'gamma': [0, 0.1, 0.2, 0.3, 0.4],
    'min_child_weight': [1, 3, 5, 7],
    'reg_alpha': [0, 0.01, 0.1, 1.0],
    'reg_lambda': [0.1, 1.0, 10.0]
}

print('XGBoost hyperparameter grid defined')

XGBoost hyperparameter grid defined


In [12]:
from xgboost import XGBRegressor


def perform_grid_search(model, param_grid, model_name, X_train, y_train, cv=3, use_randomized=False):
    """
    Perform grid search or randomized search for hyperparameter tuning
    """
    print(f"\nTuning {model_name}...")
    
    if use_randomized:
        search = RandomizedSearchCV(
            model, param_grid, 
            n_iter=50, 
            cv=cv, 
            scoring='neg_mean_squared_error',
            n_jobs=-1, 
            random_state=42,
            verbose=1
        )
    else:
        search = GridSearchCV(
            model, param_grid, 
            cv=cv, 
            scoring='neg_mean_squared_error',
            n_jobs=-1, 
            verbose=1
        )
    
    search.fit(X_train, y_train)
    
    print(f"Best parameters: {search.best_params_}")
    print(f"Best CV score (neg MSE): {search.best_score_:.4f}")
    
    return search.best_estimator_, search.best_params_, search.best_score_

# Perform XGBoost hyperparameter tuning
print("\\nTuning XGBoost...")

xgb_model = XGBRegressor(random_state=42, verbosity=0, n_jobs=-1)

search = RandomizedSearchCV(
    xgb_model, xgb_param_grid, 
    n_iter=100,  # More iterations for better search
    cv=5,  # 5-fold CV
    scoring='neg_mean_squared_error',
    n_jobs=-1, 
    random_state=42,
    verbose=2
)

search.fit(X_train, y_train)

print(f"Best parameters: {search.best_params_}")
print(f"Best CV score (neg MSE): {search.best_score_:.4f}")

tuned_xgb = search.best_estimator_
xgb_params = search.best_params_
xgb_score = search.best_score_

\nTuning XGBoost...
Fitting 5 folds for each of 100 candidates, totalling 500 fits
[CV] END colsample_bytree=0.7, gamma=0.2, learning_rate=0.2, max_depth=3, min_child_weight=7, n_estimators=500, reg_alpha=1.0, reg_lambda=0.1, subsample=0.8; total time=   0.5s
[CV] END colsample_bytree=0.7, gamma=0.2, learning_rate=0.2, max_depth=3, min_child_weight=7, n_estimators=500, reg_alpha=1.0, reg_lambda=0.1, subsample=0.8; total time=   0.6s
[CV] END colsample_bytree=0.7, gamma=0.2, learning_rate=0.2, max_depth=3, min_child_weight=7, n_estimators=500, reg_alpha=1.0, reg_lambda=0.1, subsample=0.8; total time=   0.5s
[CV] END colsample_bytree=0.7, gamma=0.2, learning_rate=0.2, max_depth=3, min_child_weight=7, n_estimators=500, reg_alpha=1.0, reg_lambda=0.1, subsample=0.8; total time=   0.5s
[CV] END colsample_bytree=0.7, gamma=0.2, learning_rate=0.2, max_depth=3, min_child_weight=7, n_estimators=500, reg_alpha=1.0, reg_lambda=0.1, subsample=0.8; total time=   0.5s
[CV] END colsample_bytree=0.7, g

In [ ]:
# Evaluate tuned XGBoost on test set
def evaluate_model(model, model_name, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    metrics = {
        'model': model_name,
        'train_rmse': mean_squared_error(y_train, y_train_pred),
        'test_rmse': mean_squared_error(y_test, y_test_pred),
        'train_mae': mean_absolute_error(y_train, y_train_pred),
        'test_mae': mean_absolute_error(y_test, y_test_pred),
        'train_r2': r2_score(y_train, y_train_pred),
        'test_r2': r2_score(y_test, y_test_pred),
    }
    return metrics, y_test_pred

# Evaluate tuned XGBoost
tuned_metrics, tuned_y_pred = evaluate_model(tuned_xgb, 'XGBoost_tuned', X_train, y_train, X_test, y_test)
tuned_metrics['cv_score'] = -xgb_score  # Convert back from negative MSE

# Display results
results_df = pd.DataFrame([tuned_metrics])
results_df = results_df[['model', 'cv_score', 'train_rmse', 'test_rmse', 'train_mae', 'test_mae', 'train_r2', 'test_r2']]
results_df = results_df.round(4)
results_df

TypeError: got an unexpected keyword argument 'squared'

In [ ]:
# Evaluate tuned XGBoost vs baseline
def evaluate_model(model, model_name, X_train, y_train, X_test, y_test):
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    metrics = {
        'model': model_name,
        'train_rmse': np.sqrt(mean_squared_error(y_train, y_train_pred)),
        'test_rmse': np.sqrt(mean_squared_error(y_test, y_test_pred)),
        'train_mae': mean_absolute_error(y_train, y_train_pred),
        'test_mae': mean_absolute_error(y_test, y_test_pred),
        'train_r2': r2_score(y_train, y_train_pred),
        'test_r2': r2_score(y_test, y_test_pred),
    }
    return metrics, y_test_pred

# Evaluate tuned XGBoost
tuned_metrics, tuned_y_pred = evaluate_model(tuned_xgb, 'XGBoost_tuned', X_train, y_train, X_test, y_test)
tuned_metrics['cv_score'] = -xgb_score  # Convert back from negative MSE

# Display results
print('XGBoost Tuning Results:')
print(f'Baseline XGBoost - Test RMSE: {baseline_metrics["test_rmse"]:.4f}, R²: {baseline_metrics["test_r2"]:.4f}')
print(f'Tuned XGBoost - Test RMSE: {tuned_metrics["test_rmse"]:.4f}, R²: {tuned_metrics["test_r2"]:.4f}')
print(f'Improvement: {baseline_metrics["test_rmse"] - tuned_metrics["test_rmse"]:.4f} RMSE reduction')


In [ ]:
# Visualize tuned XGBoost predictions
print(f'Best tuned model: XGBoost_tuned')
print(f'Test RMSE: {tuned_metrics["test_rmse"]:.4f}')
print(f'Test R²: {tuned_metrics["test_r2"]:.4f}')

# Predictions vs Actual
plt.figure(figsize=(8, 8))
sns.scatterplot(x=y_test, y=tuned_y_pred, alpha=0.4)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red', linestyle='--')
plt.xlabel('Actual price_log')
plt.ylabel('Predicted price_log')
plt.title(f'Actual vs Predicted price_log (XGBoost_tuned)')
plt.tight_layout()
plt.show()

# Residual distribution
residuals = y_test - tuned_y_pred
plt.figure(figsize=(8, 4))
sns.histplot(residuals, kde=True, bins=35)
plt.title(f'Residual Distribution (XGBoost_tuned)')
plt.xlabel('Residual')
plt.tight_layout()
plt.show()


In [ ]:
# Save best XGBoost parameters
best_params = {'XGBoost': xgb_params}

# Save to JSON
params_path = processed_dir / 'best_xgb_params.json'
with open(params_path, 'w') as f:
    json.dump(best_params, f, indent=2)

print(f'Saved best XGBoost parameters to: {params_path}')
print('\nBest XGBoost Parameters:')
for param, value in xgb_params.items():
    print(f'{param}: {value}')


## Summary

This notebook performed hyperparameter tuning on XGBoost for eBay price prediction:

- **XGBoost Tuning**: Randomized search over comprehensive parameter space
- **Cross-Validation**: 5-fold CV with negative MSE scoring
- **Parameter Space**: 9 parameters with 100 random combinations

### Key Findings:
- Tuned XGBoost outperforms baseline XGBoost
- Best parameters saved to `datasets/processed/best_xgb_params.json`

### Next Steps:
- Feature selection and engineering
- Model interpretation and feature importance analysis
- Ensemble methods with other tuned models
- Production model deployment
